In [5]:
%load_ext autoreload
%autoreload 2
%load_ext rpy2.ipython

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [6]:
import ibis
import pandas as pd

import src
from src.load import DataLoader

ibis.options.interactive = True
r_colormap = src.r_colormap
r_out = str(src.OUT)
pd.options.display.float_format = "{:.1f}".format

In [7]:
%%R -i r_colormap -i r_out

suppressMessages(library(tidyverse))
library(ggplot2)
library(ggeffects)
library(here)
library(ggpubr)

options(scipen = 999)

cmap <- setNames(r_colormap$color, r_colormap$channel)

# Load Data

In [11]:
dl = DataLoader()

videos = dl.channels().join(dl.videos(filtered=True), "channel_id").to_pandas()

sentences = dl.sentences(filtered=True).join(dl.popbert(filtered=False), "sentence_id").to_pandas()

sents = sentences.groupby("video_id", observed=True).agg(
    n_sents=("video_id", "size"),
    n_elite=("elite", "sum"),
    n_pplcentr=("pplcentr", "sum"),
    avg_elite=("elite", "mean"),
    avg_pplcentr=("pplcentr", "mean"),
)

# Dataset Summary Table

In [ ]:
channel_overview = (
    videos.loc[videos.video_was_live == False]
    .merge(sents, on="video_id", how="left")
    .groupby("channel", observed=True)
    .agg(
        chFollowers=("channel_follower_count", "first"),
        nShortVideos=("video_is_short", lambda x: (x == 1).sum()),
        nLongVideos=("video_is_short", lambda x: (x == 0).sum()),
        nTotalVideos=("video_was_live", lambda x: (x == 0).sum()),
        nLikesNA=("video_likes", lambda x: x.isna().sum()),
        meanViews=("video_views", "mean"),
        medianViews=("video_views", "median"),
        meanLikes=("video_likes", "mean"),
        medianLikes=("video_likes", "median"),
        meanVideoLen=("video_duration", "mean"),
        nSentences=("n_sents", "sum"),
        first_video=("video_datetime_upload", "min"),
        latest_video=("video_datetime_upload", "max"),
    )
)

In [13]:
channel_overview

,chFollowers,nLongVideos,nShortVideos,nTotalVideos,nLikesNA,meanViews,medianViews,meanLikes,medianLikes,meanVideoLen,nSentences,first_video,latest_video
channel,,,,,,,,,,,,,
AfD BT,521000,5539,642,6181,0,53458.1,16477.0,4362.1,2053.0,411.4,374926,2017-12-06 13:23:54,2025-02-20 15:14:38
AfD TV,334000,1236,593,1829,108,53608.7,19754.0,4374.7,2553.0,558.9,172537,2017-12-08 00:21:22,2025-02-21 15:09:57
CDU,30000,320,507,827,1,20985.7,1291.0,200.7,28.0,569.3,73946,2017-12-11 16:28:36,2025-02-22 19:13:53
CSU,6610,68,104,172,4,26680.9,923.0,45.2,24.0,436.0,13608,2017-12-14 21:19:06,2025-02-19 13:55:30
FDP,28900,333,392,725,724,22122.6,1191.0,105.0,105.0,566.6,58962,2018-01-06 16:02:32,2025-02-22 16:51:16
Greens,35400,425,218,643,7,16637.4,1316.0,196.6,32.0,604.0,55008,2018-01-27 10:45:39,2025-02-22 23:09:52
Left,117000,356,392,748,3,29150.1,3505.5,1876.5,164.0,585.0,65262,2017-12-11 14:33:45,2025-02-23 16:28:05
SPD,34500,428,625,1053,2,10700.0,2435.0,192.6,85.0,594.1,93973,2017-12-07 12:17:56,2025-02-23 20:14:54


In [14]:
inlines = src.OUT / "manuscript/inlines"
inlines.mkdir(exist_ok=True)

In [17]:
# number of broken transcripts

broken_transcripts = dl.broken_transcripts(filtered=True).to_pandas()
n_broken = len(broken_transcripts)

n_broken = f"{n_broken:,.0f}"
p = inlines / "n_broken.txt"
p.unlink(missing_ok=True)
p.write_text(n_broken)
print(f"faulty transcripts: {n_broken}")

faulty transcripts: 80


In [18]:
# sum of durations

sum_of_seconds = videos.video_duration.sum()
hour_duration = f"{round(sum_of_seconds / 60 / 60, 1):,.1f}"
p = inlines / "sum_duration.txt"
p.unlink(missing_ok=True)
p.write_text(hour_duration)
print(f"Total sum of video durations: {hour_duration} hours")

Total sum of video durations: 1,659.2 hours


In [19]:
# number of videos

count_videos = len(videos)
count_videos = f"{count_videos:,.0f}"
p = inlines / "n_videos.txt"
p.unlink(missing_ok=True)
p.write_text(count_videos)
print(f"Total number of valid videos: {count_videos}")

Total number of valid videos: 12,178


In [20]:
# number of sentencs

count_sents = len(sentences)
count_sents = f"{count_sents:,.0f}"
p = inlines / "n_sents.txt"
p.unlink(missing_ok=True)
p.write_text(count_sents)
print(f"Total number of valid sentences: {count_sents}")

Total number of valid sentences: 908,222


In [21]:
summary_table = channel_overview.drop(["first_video", "latest_video"], axis=1).T

summary_table

channel,AfD BT,AfD TV,CDU,CSU,FDP,Greens,Left,SPD
chFollowers,521000.0,334000.0,30000.0,6610.0,28900.0,35400.0,117000.0,34500.0
nLongVideos,5539.0,1236.0,320.0,68.0,333.0,425.0,356.0,428.0
nShortVideos,642.0,593.0,507.0,104.0,392.0,218.0,392.0,625.0
nTotalVideos,6181.0,1829.0,827.0,172.0,725.0,643.0,748.0,1053.0
nLikesNA,0.0,108.0,1.0,4.0,724.0,7.0,3.0,2.0
meanViews,53458.1,53608.7,20985.7,26680.9,22122.6,16637.4,29150.1,10700.0
medianViews,16477.0,19754.0,1291.0,923.0,1191.0,1316.0,3505.5,2435.0
meanLikes,4362.1,4374.7,200.7,45.2,105.0,196.6,1876.5,192.6
medianLikes,2053.0,2553.0,28.0,24.0,105.0,32.0,164.0,85.0
meanVideoLen,411.4,558.9,569.3,436.0,566.6,604.0,585.0,594.1


In [22]:
path = src.OUT / "tables/dataset_summary.csv"
path.unlink(missing_ok=True)
summary_table.to_csv(path)

# View Count Violin Plot

In [23]:
df = videos.merge(sents, on="video_id")

In [17]:
%%R -i df -w 1000 -h 800

df_plot <- df %>%
   mutate(
      likes = video_likes + 1,
      views = video_views + 1,
)

view_plot = ggplot(df_plot, aes(x=channel, y=views, fill=channel)) +
   geom_boxplot(alpha=0.6) +
   geom_violin(alpha=0.3, trim=T, scale="width") +
   scale_y_continuous(trans=scales::log10_trans(), breaks=scales::breaks_log(n=8)) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 21
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(ViewCount)")

like_plot = ggplot(df_plot, aes(x=channel, y=likes, fill=channel)) +
   geom_boxplot(alpha=0.6) +
   geom_violin(alpha=0.3, trim=T, scale="width") +
   scale_y_continuous(trans=scales::log10_trans(), breaks=scales::breaks_log(n=8)) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 21
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(LikeCount)")


ggarrange(view_plot, like_plot, ncol=2)

p <- here(r_out, "/figures/view_count.svg")
if (file.exists(p)) file.remove(p)
ggsave(p, width=12.4, height=9)

In addition: Warning messages:
1: Removed 849 rows containing non-finite outside the scale range
(`stat_boxplot()`). 
2: Removed 849 rows containing non-finite outside the scale range
(`stat_ydensity()`). 
3: Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes. 


# Populism Amount Plot

In [18]:
%%R -i df -w 1000 -h 800

df_plot <- df %>%
   mutate(
      elite = (n_elite / n_sents * 100) + 1,
      pplcentr = (n_pplcentr / n_sents * 100) + 1,
)

elite_plot = ggplot(df_plot, aes(x=channel, y=elite, fill=channel)) +
   geom_boxplot(alpha=0.6, outliers=F, coef=0.5) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("% Anti-Elitism")

pplcentr_plot = ggplot(df_plot, aes(x=channel, y=pplcentr, fill=channel)) +
   geom_boxplot(alpha=0.6, outliers=F, coef=0.5) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("% People-Centrism")


ggarrange(elite_plot, pplcentr_plot, ncol=2)

p <- here(r_out, "/figures/populism_per_party.svg")
if (file.exists(p)) file.remove(p)
ggsave(p, width=12.4, height=9.0)